In [2]:
# test & print the library version
import numpy as np
import cv2
import mediapipe as mp


print(f"NumPy 版本: {np.__version__}")        # 1.26.4
print(f"OpenCV 版本: {cv2.__version__}")      # 4.10.0
print(f"Mdeiapipe 版本: {mp.__version__}")    # 0.10.21


NumPy 版本: 1.26.4
OpenCV 版本: 4.10.0
Mdeiapipe 版本: 0.10.21


In [6]:
#game_1 : "find_this_mii"

import cv2
import numpy as np

# function 
def mouse_callback(event, _x, _y, flags, param):
    global show_img, x0, y0, w, h, mouse_pressed
    if event == cv2.EVENT_LBUTTONDOWN:
        mouse_pressed = True
        x0, y0 = _x, _y
        show_img = np.copy(cur_frame)
    elif event == cv2.EVENT_MOUSEMOVE:
        if mouse_pressed:
            show_img = np.copy(cur_frame)
            cv2.rectangle(show_img, (x0, y0), (_x, _y), (0, 255, 0), 2)
    elif event == cv2.EVENT_LBUTTONUP:
        mouse_pressed = False
        w, h = _x - x0, _y - y0

#main
cv2.namedWindow('image')
cv2.setMouseCallback('image', mouse_callback)

scaling_factor = 0.5
cap = cv2.VideoCapture("WiiPlay.mp4")
frame_seq = 4820

'''
Acquire a face template from the first frame 
'''
cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
ret, frame = cap.read()
cur_frame = cv2.resize(frame, None, fx=scaling_factor, fy=scaling_factor)
show_img = np.copy(cur_frame)
mouse_pressed = False
y0 = x0 = w = h = 0

while True:
    cv2.imshow('image', show_img)
    k = cv2.waitKey(1)
    #chr(97) => 'a'
    #ord('a') => 97
    if k == ord('a') and not mouse_pressed:
        if w*h > 0:
            break
cv2.destroyAllWindows()

template = np.copy(cur_frame[y0:y0+h, x0:x0+w])
height, width = template.shape[0:2]

# 定義縮放範圍 包含 0.8, 0.9, 1.0, 1.1, 1.2
scales = np.arange(0.8, 1.3, 0.1)
method = 'cv2.TM_SQDIFF'

'''
detect the face the same as the template
'''
while True:
    frame_seq += 1
    if frame_seq > 5000:
        frame_seq = 4820
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()        

    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)

    # 初始化最佳匹配結果
    best_match_val = float('inf')
    best_match_loc = None
    best_match_w = 0
    best_match_h = 0
    best_res_map = None # 儲存最佳匹配的結果圖

    for scale in scales:
        # 縮放模板
        resized_template = cv2.resize(template, None, fx=scale, fy=scale)
        h_r, w_r = resized_template.shape[:2]
        if h_r > frame.shape[0] or w_r > frame.shape[1]:
            continue

        res = cv2.matchTemplate(frame, resized_template, eval(method))
        #print('min:{} max:{}'.format(np.min(res), np.max(res)))
        res = cv2.normalize(res, None, 0, 1, cv2.NORM_MINMAX)            
        #print('min_n:{} max_n:{}'.format(np.min(res), np.max(res)))
        min_val , max_val , min_loc , max_loc , = cv2 . minMaxLoc ( res )
        # 比較並更新最佳結果 (尋找最小值)
        if min_val < best_match_val:
            best_match_val = min_val
            best_match_loc = min_loc
            best_match_w = w_r
            best_match_h = h_r
            best_res_map = res # 儲存此尺度的結果圖
            
    # 繪製最佳匹配結果
    if best_match_loc is not None :
        # 繪製在原始幀上 (使用最佳匹配的尺寸)
        pt = best_match_loc
        res_img = best_res_map
        cv2 . rectangle ( frame , pt , ( pt [ 0 ] + best_match_w , pt [ 1 ] + best_match_h ) , ( 0 , 0 , 255 ) , 3 )
        cv2 . rectangle ( res_img , pt , ( pt [ 0 ] + best_match_w , pt [ 1 ] + best_match_h ) , ( 0 , 0 , 255 ) , 3 )

    
    res_img = cv2.resize(res_img, (640, 360))*255
    res_img = cv2.cvtColor(res_img, cv2.COLOR_GRAY2BGR).astype(np.uint8)
    #print("1: ", res_img.shape)
    #print("2: ", frame.shape)
    frame = np.hstack((frame, res_img))
    cv2.imshow("find_this_mii", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

程式上大致參考11/26 opencv_template練習
主要是額外加上了resize template的for loop
來處理進入遊戲後人物比開始時的目標圖示小與滑鼠靠近後變大的問題

再來就是依據match分數，只讓最高分的出現，讓每個frame只出現1個框框

最後結果雖然沒到完美，但至少6～7成的frame是可以正確抓到的
finish A,B,C(60~70%)

In [1]:
#game_2 : "find_two_look_alike"

import cv2
import numpy as np
from ultralytics import YOLO
import mediapipe as mp

scaling_factor = 0.5
cap = cv2.VideoCapture("WiiPlay.mp4")
frame_seq = 2180

model = YOLO('yolov8n.pt')
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5)

while True:
    frame_seq += 1
    if frame_seq > 2380:
        frame_seq = 2180
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()
    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)

    # for part 2B detect pedestrians by using yolo
    results = model.predict(frame, conf=0.25, verbose=False)
    person_in_frame = 0
    persons = {}
    template = {}
    face_data = {}
    
    for result in results:
        for box in result.boxes:
            label = result.names[int(box.cls)]
            if label != "person":
                continue
            #print(box.xyxy) #box.xyxy.cpu()
            left, top, right, bottom = np.array(box.xyxy, dtype=np.uint16).squeeze() #convert from tensor to list
            #print(left, top, right, bottom)
            confidence = float(box.conf.cpu())
            cv2.rectangle(frame, (left, top),(right, bottom), (0, 255, 0), 2)
            #cv2.putText(frame, label+' '+'{:.2f}'.format(confidence),(left+5, bottom-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1, cv2.LINE_AA)
            
            persons[person_in_frame] = frame[top:bottom, left:right]
            person_in_frame += 1

            person_rgb = cv2.cvtColor(persons[person_in_frame-1], cv2.COLOR_BGR2RGB)
            results = face_detector.process(person_rgb)
            if results.detections:
                for detection in results.detections:
                    bbox_c = detection.location_data.relative_bounding_box
                    ih, iw, _ = persons[person_in_frame-1].shape
                    x = int(bbox_c.xmin * iw) + left
                    y = int(bbox_c.ymin * ih) + top
                    w = int(bbox_c.width * iw)
                    h = int(bbox_c.height * ih)
                    cv2.rectangle(frame, (x,y), (x+w,y+h), (255, 0, 0), 2)
                    template[person_in_frame-1] = frame[y:y+h, x:x+w]
                    face_data[person_in_frame-1] = (x, y, w, h)

    # Part D
    max_similarity = -1.0
    best_match_indices = None
    
    face_indices = list(template.keys())
    
    # 確保至少有兩張臉可以比對
    if len(face_indices) >= 2:
        
        for i in range(len(face_indices)):
            for j in range(i + 1, len(face_indices)):
                
                idx1 = face_indices[i]
                idx2 = face_indices[j]
                
                template1 = template[idx1]
                template2 = template[idx2]
                
                h1, w1, _ = template1.shape
                h2, w2, _ = template2.shape
                
                if w1 * h1 < w2 * h2:
                    template_small = template1
                    image_large = template2
                else:
                    template_small = template2
                    image_large = template1
                try:
                    resized_template2 = cv2.resize(template2, (w1, h1))
    
                    gray_template1 = cv2.cvtColor(template1, cv2.COLOR_BGR2GRAY)
                    gray_template2 = cv2.cvtColor(resized_template2, cv2.COLOR_BGR2GRAY)
    
                    result_matrix = cv2.matchTemplate(gray_template1, gray_template2, cv2.TM_CCOEFF_NORMED)
                    similarity = result_matrix[0, 0]
                        
                    if similarity > max_similarity:
                        max_similarity = similarity
                        best_match_indices = (idx1, idx2)
                except cv2.error as e:
                    continue
    if best_match_indices and max_similarity >= 0.7:
        idx1, idx2 = best_match_indices
        (x1, y1, w1, h1) = face_data[idx1]
        (x2, y2, w2, h2) = face_data[idx2]
        
        cv2.rectangle(frame, (x1, y1), (x1 + w1, y1 + h1), (0, 0, 255), 3)
        cv2.rectangle(frame, (x2, y2), (x2 + w2, y2 + h2), (0, 0, 255), 3)

    
    cv2.imshow("find_two_look_alike", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
face_detector.close()
cv2.destroyAllWindows()

I0000 00:00:1766932336.158645   66229 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1766932336.161469   66279 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) Graphics (ADL-N)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1766932336.169058   66273 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


### PartB
原本想說利用'HOGDescriptor'來去找，可惜效果不佳

因為人物在被遮擋或是沒有整個人出現在畫面時是偵測不到的

後來想到12/17 YOLO的範例程式，即使是只有一隻手，YOLO也可以正確偵測到這是個人，並標示出來

再來就是'model.predict'時加上參數'verbose=False'，關閉每幀的log
### PartC
之後在依據YOLO標出的行人，藉由'haarcascade_frontalface_default.xml'找出人臉

可惜依舊效果不佳

再次改用'mediapipe.solutions.face_detection'，效果就好上不少

### PartD
想法是將前面的person, face儲存成陣列，之後在用'matchTemplate'處理

finish: A,B,C

In [17]:
#game_3 : "find_the_fastest_character"

import cv2
import numpy as np
from ultralytics import YOLO

scaling_factor = 0.5
cap = cv2.VideoCapture("WiiPlay.mp4")
frame_seq = 2480 # 從原本2480改成2485，因為第一幀讀不到最快的那個角色

model = YOLO('yolov8n.pt')

cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
status_cap, frame0 = cap.read()
frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)


# for Part B 
results = model.predict(frame, conf=0.25, verbose=False)
person_in_frame = 0
persons = {}
person_data = []
    
for result in results:
    for box in result.boxes:
        label = result.names[int(box.cls)]
        if label != "person":
            continue
        #print(box.xyxy) #box.xyxy.cpu()
        left, top, right, bottom = np.array(box.xyxy, dtype=np.uint16).squeeze() #convert from tensor to list
        #print(left, top, right, bottom)
        confidence = float(box.conf.cpu())
        cv2.rectangle(frame, (left, top),(right, bottom), (255, 0, 0), 2)
        #cv2.putText(frame, label+' '+'{:.2f}'.format(confidence),(left+5, bottom-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1, cv2.LINE_AA)
            
        persons[person_in_frame] = frame[top:bottom, left:right]
        position = (left, top, right - left, bottom - top)
        person_data.append(position)
        person_in_frame += 1

cv2.imshow("first frame", frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Part C
trackers = []
for person in person_data:
    t = cv2.TrackerMIL_create()
    t.init(frame, person)
    trackers.append(t)

while True:
    frame_seq += 1
    if frame_seq > 2600:
        frame_seq = 2480
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()
    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)
    
    for t in trackers:
        status_tracker, bbox = t.update(frame)
        if status_tracker:
            x, y, w, h = [int(i) for i in bbox]
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    
    cv2.imshow("find_the_fastest_character", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

### Part B
同第二題，利用YOLO尋找行人

### Part C 
參考12/24 opencv_motion中的MIL_tracker完成行人追蹤

但有個問題MIL_tracker會因為丟失或重疊等原因，追蹤不到正確的位置

finish: A,B,C

In [36]:
#game_4 : "find_two_odds"

import cv2
import numpy as np

def display_flow(img, flow, stride=10):
    #count = 0
    for index in np.ndindex(flow[::stride, ::stride].shape[:2]):
        pt1 = tuple(i*stride for i in index)
        delta = flow[pt1].astype(np.int32)[::-1]
        pt2 = tuple(pt1 + 2*delta)
        if 2 <= cv2.norm(delta) <= 10:
            #count = count + 1
            cv2.arrowedLine(img, pt1[::-1], pt2[::-1], (255,0,0), 1, cv2.LINE_AA, 0, 0.1)
        #print(count)
    norm_opt_flow = np.linalg.norm(flow, axis=2)
    norm_opt_flow = cv2.normalize(norm_opt_flow, None, 0, 1, cv2.NORM_MINMAX)
    cv2.imshow('find_two_odds', img)
    cv2.imshow('optical flow magnitude', norm_opt_flow)


scaling_factor = 0.5
cap = cv2.VideoCapture("WiiPlay.mp4")
frame_seq = 1650

cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
_ , prev_frame = cap.read()
prev_frame = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_frame = cv2.resize(prev_frame, None, fx=scaling_factor, fy=scaling_factor)
first_frame = True


while True:
    frame_seq += 4
    if frame_seq > 1800:
        frame_seq = 1650
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()
    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    if first_frame:
        opt_flow = cv2.calcOpticalFlowFarneback(prev_frame, gray, None, 0.5, 5, 15, 10, 5, 1.1, cv2.OPTFLOW_FARNEBACK_GAUSSIAN)
        first_frame = False
    else:
        opt_flow = cv2.calcOpticalFlowFarneback(prev_frame, gray, opt_flow, 0.5, 5, 15, 10, 5, 1.1, cv2.OPTFLOW_USE_INITIAL_FLOW)
    prev_frame = np.copy(gray)
    display_flow(frame, opt_flow)
    
    cv2.imshow("find_two_odds", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

### Part B
如果是每幀比對臉轉的速度太慢，箭頭不夠明顯，所以便把frame_seq變化速度(從+1變+4)加快，就能很明顯的看出箭頭

finish: A,B

In [48]:
#game_5: Licence Plate Recognition
# install tesseract.exe from https://github.com/UB-Mannheim/tesseract/wiki
# pip install pytesseract
# pip install ultralytics

from ultralytics import YOLO
import cv2
import pytesseract

# load models
model = YOLO('yolo12n.pt')
license_plate_detector = YOLO('license_plate_detector.pt')

# Path to the Tesseract executable
# in windows
#pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
# in linux
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'


scaling_factor = 0.3
cap = cv2.VideoCapture("car.mp4")
frame_seq = 0

while True:
    frame_seq += 1
    if frame_seq > 1700:
        frame_seq = 0
    cap.set(cv2.CAP_PROP_POS_FRAMES , frame_seq)
    status_cap, frame0 = cap.read()
    if not status_cap:
        break
    frame = cv2.resize(frame0, None, fx=scaling_factor, fy=scaling_factor)

    results = model.predict(frame, conf=0.25, verbose=False)
    for result in results:
        for box in result.boxes:
            label = result.names[int(box.cls)]
            if label != "car":
                continue
            #print(box.xyxy) #box.xyxy.cpu()
            left, top, right, bottom = np.array(box.xyxy, dtype=np.uint16).squeeze() #convert from tensor to list
            #print(left, top, right, bottom)
            confidence = float(box.conf.cpu())
            cv2.rectangle(frame, (left, top),(right, bottom), (0, 0, 255), 2)
            #cv2.putText(frame, label+' '+'{:.2f}'.format(confidence),(left+5, bottom-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 1, cv2.LINE_AA)
    
            license_plates = license_plate_detector(frame[top:bottom, left:right], verbose=False)[0]
        
            for license_plate in license_plates.boxes.data.tolist():
                x1, y1, x2, y2, score, class_id = license_plate
                x1 += left
                x2 += left
                y1 += top
                y2 += top
                #print(x1, y1, x2, y2)
            
                # crop license plate
                license_plate_crop = frame[int(y1):int(y2), int(x1): int(x2), :]
            
                # process license plate
                license_plate_crop_gray = cv2.cvtColor(license_plate_crop, cv2.COLOR_BGR2GRAY)
                _, license_plate_crop_thresh = cv2.threshold(license_plate_crop_gray, 64, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
            
                # Recognize text from the license plate
                text = pytesseract.image_to_string(license_plate_crop_thresh, config='--psm 7') #--psm 11
                text = text.upper().replace(' ', '')
                #print("Detected license plate Number is:", text.strip())
            
                # Draw a rectangle around the license plate
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
                cv2.putText(frame, text.strip(), (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36,255,12), 2)

    
    cv2.imshow("Licence Plate Recognition", frame)
    k = cv2.waitKey(1)
    if k == 27:
        break
        
cap.release()
cv2.destroyAllWindows()

finish: A,B,C,D

###### %%writefile test
## Final Exam
<b>The general objective is to solve five different games.<br>
The code for each game should be written in an individual cell.<br>
Add comments in your code to explain your approach.<br>
You can design your own approach, and use any method you learned in this class.</b><br>

1A. Input images from video file WiiPlay.mp4 with level 15 (frame number between 4820 and 5000).<br> 
1B. (5pts) Acquire a <b>face template</b> from the first frame (frame number = 4820).<br>
1C. (10pts) Try to detect the face the same as the template on subsequent frames, draw a <b>red</b> rectangle around the detected face, and show the output images in the <b>"find_this_mii"</b> window.<br><br>

2A. Input images from video file WiiPlay.mp4 with level 8 (frame number between 2180 and 2380).<br>
2B. (5pts) Detect <b>pedestrians</b> on each frame and draw a <b>green</b> rectangle around your detection.<br>
2C. (5pts) Detect <b>faces</b> on each frame and draw a <b>blue</b> rectangle around your detection.<br>
2D. (10pts) Try to find two faces look like each other, draw a <b>red</b> rectangle around each of the two faces, and show the output images in the <b>"find_two_look_alike"</b> window.<br><br>

3A. Input images from video file WiiPlay.mp4 with level 9 (frame number between 2480 and 2600).<br>
3B. (5pts) <b>Detect </b>faces(or pedestrians) on the first frame and draw a <b>blue</b> rectangle around your detection.<br>
3C. (10pts) <b>Track </b>faces(or pedestrians) on subsequent frames and draw a <b>green</b> rectangle around your tracking.<br>
3D. (5pts) Try to find out the fastest character, draw a <b>red</b> rectangle around the fastest character, and show the output images in the <b>"find_the_fastest_character"</b> window.<br><br>

4A. Input images from video file WiiPlay.mp4 with level 6 (frame number between 1650 and 1800).<br>
4B. (10pts) Compute and show <b>optical flows</b> on each frame using <b>blue</b> arrows.<br>
4C. (5pts) Try to detect two odd character who face the opposite direction from everyone else, draw a <b>red</b> rectangle around each of the two character, and show the output images in the <b>"find_two_odds"</b> window.<br><br>

5A. Input continuous images from 'car.mp4'.<br>
5B. (6pts) For each frame, detect every car using YOLO trained data 'yolo12n.pt'. (mark with red rectangles)<br>
5C. (6pts) For each car, detect a licence plate using 'license_plate_detector.pt'. (mark with blue rectangle)<br>
5D. (6pts) For each licence plate, OCR using Tesseract. Print the recognized licence plate number above each detected licence plate. (putText() in green color)<br>
5E. (12pts) Use whatever you learned this semester to improve the result. Write a simple report on your method and observations.<br><br>

6. (5pts) Any comments regarding the final exam? Which steps you believe you have completed? Which steps bother you?<br>
7. (5pts) Any suggestion to teaching assistants to improve this class? Any suggestion to teacher to improve this class?<br>
8. Upload your Jupyter file (*.ipynb) with code (for 1-5) and report (for 1-7). 

---
#### 6 Any comments regarding the final exam? Which steps you believe you have completed? Which steps bother you?

關於完成哪些與碰到哪些問題，已附在程式碼後一個cell



---
#### 7 Any suggestion to teaching assistants to improve this class? Any suggestion to teacher to improve this class?

之前上課時有遇到一個蠻特別的狀況，就是mediapipe剛好更新，導致很多指令要改

目前在自己的電腦測下來
* NumPy 版本: 1.26.4（須降版）
* OpenCV 版本: 4.10 （須降版）
* Mdeiapipe 版本: 0.10.21

Tesseract/YOLO/Mediapipe
都是可以正常使用的，且指令上沒什麼差異

安裝完如果發生版本衝突都可以直接降到此版本